In [3]:
from __future__ import annotations

import importlib.util
import os
import platform
import statistics
import subprocess
import sys
import sysconfig
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

MAX_WORKERS = 2
DATA_DIR = Path.cwd() / "aula2_dados_sinteticos"

print("Diretório de trabalho: ", Path.cwd())
print("MAX_WORKERS: ", MAX_WORKERS)


Diretório de trabalho:  /home/gusta/Área de trabalho/IA_FATESG3/COMPUTACAO_PARALELA/Cp_Aula2
MAX_WORKERS:  2


In [7]:
#Diagnostico do Ambiente

def diagnosticar_ambiente() -> dict:
    cpu_count_fn = getattr(os, "process_cpu_count", os.cpu_count)
    cpu_count = cpu_count_fn() or 1
    supports_ft = sysconfig.get_config_var("Py_GIL_DISABLED") == 1
    gil_check = getattr(sys, "_is_gil_enabled", None)
    gil_enabled = gil_check() if callable(gil_check) else None

    try:
        has_google_colab = importlib.util.find_spec("google.colab") is not None
    except (ModuleNotFoundError, ImportError):
        has_google_colab = False

    ambiente = "Google Colab" if has_google_colab else "Jupyter/VSCode ou outro"
    return {
        "ambiente": ambiente,
        "implementacao": platform.python_implementation(),
        "python": platform.python_version(),
        "cpus_logicas_visiveis": cpu_count,
        "build_free_threaded": supports_ft,
        "gil_ativo_detectavel": gil_enabled,
    }

info = diagnosticar_ambiente()
for chave, valor in info.items():
    print(f"{chave}: {valor}")

ambiente: Jupyter/VSCode ou outro
implementacao: CPython
python: 3.12.3
cpus_logicas_visiveis: 4
build_free_threaded: False
gil_ativo_detectavel: None


In [9]:
def criar_arquivos_sinteticos(diretorio: Path, quantidade: int = 8, tamanho: int = 4096) -> list[Path]:
    diretorio.mkdir(parents=True, exist_ok=True)
    for antigo in diretorio.glob("amostra_*.bin"):
        antigo.unlink()
    
    caminhos = []
    for indice in range(quantidade):
        dados = bytes((indice * 17 + offset * 31) % 256 for offset in range(tamanho))
        caminho = diretorio / f"amostra_{indice:02d}.bin"
        caminho.write_bytes(dados)
        caminhos.append(caminho)
    return caminhos

caminhos = criar_arquivos_sinteticos(DATA_DIR)
print(f"Arquivos criados: {len(caminhos)}")
print("Primeiros:", [p.name for p in caminhos[:3]])
print(f"Tamanho do primeiro arquivo: {caminhos[0].stat().st_size} bytes")
    

Arquivos criados: 8
Primeiros: ['amostra_00.bin', 'amostra_01.bin', 'amostra_02.bin']
Tamanho do primeiro arquivo: 4096 bytes


In [13]:
def medir(funcao, repeticoes: int = 3):
    # Executa a função algumas vezes e devolve (último_resultado, mediana_em_segundos).
    tempos = []
    resultado = None
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = funcao()
        tempos.append(time.perf_counter() - inicio)
    return resultado, statistics.median(tempos)

In [11]:
def read_sample(path: Path) -> tuple[str, bytes]:
    # Representa uma leitura com espera de armazenamento/rede.
    time.sleep(0.08)
    return path.name, path.read_bytes()

def cpu_signature(item: tuple[str, bytes], rounds: int = 1200) -> tuple[str, int]:
    # Carga CPU-bound pura em Python. Mesma entrada -> mesma saída.
    name, data = item
    acc = 2166136261
    for _ in range(rounds):
        for byte in data[:512]:
            acc ^= byte
            acc = (acc * 16777619) & 0xFFFFFFFF
    return name, acc


In [15]:
io_seq, t_io_seq = medir(lambda: [read_sample(caminho) for caminho in caminhos])
def executar_io_thread():
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return list(executor.map(read_sample, caminhos))

io_thread, t_io_thread = medir(executar_io_thread)
assert io_seq == io_thread

print(f"I/O sequencial: {t_io_seq:.3f} s")
print(f"I/O threads (2 workers): {t_io_thread:.3f} s")
print("Equivalência confirmada.")

I/O sequencial: 0.643 s
I/O threads (2 workers): 0.324 s
Equivalência confirmada.
